In [1]:
import numpy as np
import scipy.io.wavfile as wavfile
import torch
from scipy.io import wavfile
from training.training_process import create_training_config

config = create_training_config()

model = config.model.generator

point = 3

model.load_state_dict(torch.load(f"./checkpoints/checkpoint-{point}000.pth", map_location=config.device))
model.eval()  # CRITICAL: Sets dropout/batchnorm to inference mode


def generate_speech(text, output_path="output.wav"):
    print(f"Generating speech for: '{text}'")

    # 1. Tokenize
    encoding = config.tokenizer(text, return_tensors="pt")
    input_ids = encoding["input_ids"].to(config.device)

    # 2. Generate (training=False triggers duration prediction)
    with torch.no_grad():
        output = model(input_ids, training=False)
    audio_tensor = output['audio_generated']
    durs = output['durs']
    print(f"Final durations: {durs.squeeze().cpu().numpy()}")
    print(f"Total mel frames: {durs.sum().item()}")

    # 3. Post-process audio
    # audio_tensor is [1, 1, Time]. Squeeze to [Time]
    audio_np = audio_tensor.squeeze().cpu().numpy()

    # Ensure it's in [-1, 1] range, then scale to 16-bit PCM
    audio_np = np.clip(audio_np, -1.0, 1.0)
    audio_int16 = (audio_np * 32767).astype(np.int16)

    # 4. Save to WAV
    sample_rate = 16000
    wavfile.write(output_path, sample_rate, audio_int16)
    print(f"✅ Successfully saved to: {output_path}")


test_text = "Привет! Это тест синтезированной речи на чистой архитектуре PyTorch."
generate_speech(test_text, f"my_first_tts_output_{point}.wav")

Generating speech for: 'Привет! Это тест синтезированной речи на чистой архитектуре PyTorch.'
Final durations: [6 2 2 1 1 2 2 2 2 2 1 2 1 2 1 2 2 2 2 1 1 2 2 2 2 2 1 1 1 2 1 2 1 1 2 1 1
 2 2 2 1 2 2 1 1 2 1 1 1 1 1 2 2 2 2 2 2 2 1 1 2 2 1 2 1 2 1 2 3 2 2 1 1 1
 1 2 1 1 1 2 3 2 2 2 1 1 2 2 1 2 2 2 1 2 1 1 2 2 2 2 1 2 1 2 2 2 2 2 2 2 1
 1 1 2 2 2 1 2 1 1 1 1 1]
Total mel frames: 200
✅ Successfully saved to: my_first_tts_output_3.wav
